# IPP Assessment Analysis

In [ ]:
%load_ext autoreload
%autoreload 3

import analysis as da
import dataset_parser

da.configure_plot_style()


In [ ]:
dataset_parser.main()


In [ ]:
df = da.load_clean_data()
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
df.head()

In [ ]:
proj = da.build_project_df(df)
print(f"projects dataframe shape: rows: {proj.shape[0]}, cols: {proj.shape[1]}")

## 1. Overview

In [ ]:
da.analyse_volume(df, proj=proj)

### Document format

PDF option was introduced in 2018, before that all submissions were in Markdown.

In [ ]:
ax = da.visualise_doc_type_distribution(df, proj=proj)

How does it affect the scores?

In [ ]:
format_impact_df = da.analyse_format_impact(df, proj=proj)
ax = da.visualise_format_impact(format_impact_df)
da.summarise_format_impact(format_impact_df)


### Task variants

The `php`/`py` labels were renamed to `par`/`int` in 2017.

2017-par subdataset missing

In [ ]:
ax = da.visualise_task_variant_distribution(df, proj=proj)

## 2. Code frequency

In [ ]:
ax = da.visualise_code_frequency(df, n_codes=25)


## 3. Impact

Each code can carry a point deduction (shown as normalised to a fraction of the year and variant maximum). 

Shared penalties (e.g. `STRUCT/SHORT`) are excluded.

### Normalised impacts

Each code's impact is expressed as a fraction of the year/variant documentation maximum, so values are comparable across years regardless of changes. The table is sorted by **mean penalty ascending**, the top rows are the harshest codes on average. The `pct_no_impact` column shows how often a code appeared as a zero-impact warning rather than an actual deduction.


In [ ]:
impact_stats_norm = da.analyse_impact_statistics(df)

print("Top 10 codes by mean penalty:")
display(impact_stats_norm.head(10))

print("\nCodes that are almost always applied as actual deductions:")
display(
    impact_stats_norm[
        (impact_stats_norm["pct_no_impact"] < 0.10) & (impact_stats_norm["count"] >= 10)
    ].sort_values("count", ascending=False)
)


In [ ]:
ax = da.visualise_impact_boxplots(df, n_codes=15)


In [ ]:
ax = da.visualise_shared_vs_individual_impact(df)

### Warning vs penalty rate

How often do graders use individual codes as warnings (with zero specified impact) vs as an actual deduction?

In [ ]:
warnings_df = da.analyse_zero_impact_warnings(df, n_codes=20)
ax = da.visualise_zero_impact_warnings(warnings_df)
da.summarise_zero_impact_warnings(warnings_df)


## 4. Trends

### Code impact severity trend

In [ ]:
g = da.visualise_code_impact_trends(df, n_codes=12)


### Code usage

Fraction of projects that received each code per year.

In [ ]:
g = da.visualise_code_usage_trends(df, n_codes=12, proj=proj)


## 5. Code relationships

### Co-occurrence

Jaccard similarity between code pairs across all projects.

In [ ]:
ax = da.visualise_code_cooccurrence(df, n_codes=15, proj=proj)


### Correlation with documentation score

Pearson correlation between code presence and normalised documentation score (% of the year/variant rubric maximum).

In [ ]:
ax = da.visualise_code_points_correlation(df, n_codes=30, proj=proj)


## 6. Comments

Graders can attach a free-text comment to any code.

### Presence

In [ ]:
comment_presence = da.analyse_comment_presence(df)

# head shows codes graders most often explain
print("Codes most frequently accompanied by a grader comment:")
display(comment_presence.head(20))

# which high-volume codes do graders almost never justify
print("\nCodes where graders rarely add a comment:")
display(
    comment_presence[
        (comment_presence["total_events"] >= 50)
        & (comment_presence["pct_commented"] <= 0.20)
    ]
)


### Length

In [ ]:
comment_length_df = da.analyse_comment_length(df)
ax = da.visualise_comment_length(comment_length_df)
comment_len_stats = da.summarise_comment_length(comment_length_df)
comment_len_stats.head(15)


### Language

In [ ]:
# takes a while thanks to langdetect
language_df = da.analyse_language_distribution(df)


In [ ]:
ax = da.visualise_language_distribution_overall(language_df)


In [ ]:
ax = da.visualise_language_distribution_by_year(language_df)


In [ ]:
lang_dist = da.summarise_language_distribution(language_df)
lang_dist


### Keywords

In [ ]:
keywords = da.analyse_comment_keywords(df, n_keywords=30)
keywords


## 7. SHORT - word and character count stats


In [47]:
from pathlib import Path

import pandas as pd
import pymupdf

DOC_BASE = Path("../data/raw/ipp_13_to_24/ipp_docs")

short_df = (
    df.query("code == 'SHORT'")[["id", "source_file", "doc_type"]]
    .drop_duplicates(["id", "source_file"])
    .reset_index(drop=True)
)

print(f"Submissions with SHORT: {len(short_df)}")
print(f"Unique student IDs: {short_df['id'].nunique()}")

records = []
for student_id, source_file, doc_type in short_df.itertuples(index=False):
    cohort = Path(source_file).stem                          # e.g. "ipp20-int"
    f = DOC_BASE / cohort / f"{student_id}.{doc_type}"

    if not f.exists():
        continue

    try:
        match doc_type:
            case "pdf":
                with pymupdf.open(f) as doc:
                    text = "".join(page.get_text() for page in doc)
            case "md":
                text = f.read_text(errors="replace")
            case _:
                continue

        records.append({
            "id": student_id,
            "cohort": cohort,
            "fmt": doc_type,
            "word_count": len(text.split()),
            "char_count": len(text),
        })
    except Exception:
        continue

wc_df = pd.DataFrame(records)
print(f"\nFiles read: {len(wc_df)} ({wc_df['fmt'].value_counts().to_dict()})")

stats = wc_df[["word_count", "char_count"]].describe(percentiles=[.1, .25, .5, .75, .9])
display(stats.round(1))


Submissions with SHORT: 1314
Unique student IDs: 1126

Files read: 1314 ({'pdf': 877, 'md': 437})


,word_count,char_count
count,1314.0,1314.0
mean,288.3,2260.3
std,171.5,8217.9
min,0.0,0.0
10%,125.0,887.2
25%,179.0,1241.2
50%,256.0,1810.5
75%,355.0,2484.8
90%,486.1,3422.5
max,2065.0,296635.0
